# PlanetScope acquisition — Folger Deep vessel-noise study

Find PlanetScope scenes that **fully cover** the 99.7 km² Folger core AOI with **minimal cloud
over that AOI**, review them for vessels using the tile quota, then order only the ones worth
paying for.

**Environment:** CryoCloud JupyterHub, ~4 GB RAM, Planet SDK v2.

## Two separate budgets

| Budget | Allocation | Cost per scene | Ceiling |
|---|---|---|---|
| **Imagery** | 3,000 km²/month | 100 km² minimum when clipping | **30 scenes/month** |
| **Scene tiles** | ~98,000 tiles/month | 56 tiles (inner box) at z15 | ~1,750 previews |

Tiles, quicklooks and UDM2 masks are all free against the imagery budget, so everything up to
stage 8 costs nothing. Only stage 8 spends, behind a manual gate.

## Selection logic

Three hard gates, then your eyes:

1. **Coverage** — the scene must contain the whole AOI
2. **Cloud** — measured on the AOI itself via free UDM2, not on the scene footprint
3. **Vessel present** — confirmed by reviewing tile previews

Whatever passes all three gets ordered, cleanest first, up to the monthly cap. There is no
weighted scoring function and no stratification by year: the gates are the selection, and
`aoi_clear` breaks ties.

## Order of operations

1. Setup and AOI
2. Search — deliberately loose
3. Coverage filter
4. UDM2 cloud screening on the AOI
5. Glint (review-queue ordering only)
6. Quicklook skim
7. Tile preview and vessel review
8. Acoustic join — **optional**
9. Select and order

## 1. Setup

In [ ]:
%pip install --quiet "planet>=2.1,<3" pandas rasterio pillow httpx shapely

In [ ]:
import asyncio, json, math, os
from datetime import datetime, timedelta
from pathlib import Path
from io import BytesIO

import numpy as np
import pandas as pd
import httpx
import rasterio
from PIL import Image
from shapely.geometry import shape
from rasterio.windows import from_bounds
from rasterio.features import geometry_mask, bounds as geom_bounds
from rasterio.warp import transform_geom
from planet import Auth, Session, data_filter, order_request, reporting

# ------------------------------------------------------------------ paths
AOI_PATH = Path("folger_core_aoi.geojson")
WORK     = Path("planet_folger");   WORK.mkdir(exist_ok=True)
UDM_DIR  = WORK / "udm2_scratch";   UDM_DIR.mkdir(exist_ok=True)
PREVIEW  = WORK / "tile_previews";  PREVIEW.mkdir(exist_ok=True)

# ------------------------------------------------------------------ search
YEARS     = range(2020, 2026)
SUMMER    = (6, 9)                 # June 1 -> Sept 1, exclusive upper bound
ITEM_TYPE = "PSScene"

# Loose on purpose. cloud_cover describes the whole ~637 km2 footprint; the AOI is
# ~15% of it. The free UDM2 screen in stage 4 measures cloud where it matters.
SEARCH_CLOUD_MAX = 0.60

# ------------------------------------------------------------------ the three gates
# 1. Coverage. Planet's geometry filter matches on INTERSECTS, not CONTAINS, and a
#    partial scene still costs the full 100 km2. Enforced as a post-filter.
MIN_COVERAGE = 0.999
# 2. Cloud, measured on the AOI via UDM2.
CLEAR_MIN    = 0.95
# 3. Vessel presence - set by eye in stage 7.

# Optional extras. Neither gates anything.
GLINT_FILTER  = None    # e.g. 20 to drop hard-glinted scenes; None = keep all,
                        # since tile budget is ample enough to just look at them
ACOUSTIC_JOIN = False   # True to cross-reference the hydrophone record
DEDUPE_BY_DAY = True    # several satellites cross the same target each morning

# ------------------------------------------------------------------ imagery quota
AOI_AREA_KM2     = 99.7
MIN_CHARGE       = 100.0
MONTHLY_QUOTA    = 3000.0
CHARGE_PER_SCENE = max(AOI_AREA_KM2, MIN_CHARGE)
SCENES_PER_MONTH = int(MONTHLY_QUOTA // CHARGE_PER_SCENE)

# ------------------------------------------------------------------ tile quota
TILE_BUDGET    = 98249
TILE_ZOOM      = 15      # 3.15 m/px here, matching PlanetScope native 3 m
PREVIEW_HALF_M = 2500    # inner box half-width; set to 5000 for the full AOI
TILE_LEDGER    = WORK / "tile_ledger.json"

# ------------------------------------------------------------------ bundle policy
# SINGLE-RASTER ONLY. A "_udm2" bundle ships a second raster asset which may draw a
# second 100 km2 minimum - for a file the Data API gives us free in stage 4.
SINGLE_RASTER_BUNDLES = {"visual", "analytic_sr", "analytic_8b_sr",
                         "analytic", "analytic_8b"}
BUNDLE = "visual"
assert BUNDLE in SINGLE_RASTER_BUNDLES, (
    f"{BUNDLE!r} is not a known single-raster bundle. Anything ending in '_udm2' "
    f"risks a doubled charge. Allowed: {sorted(SINGLE_RASTER_BUNDLES)}")
# Never set fallback_bundle: it can silently substitute a multi-raster bundle.

print(f"Gates   : coverage >= {MIN_COVERAGE:.1%}, AOI clear >= {CLEAR_MIN:.0%}, vessel visible")
print(f"Imagery : {SCENES_PER_MONTH} scenes/month at {CHARGE_PER_SCENE:.0f} km^2")
print(f"Tiles   : {TILE_BUDGET:,} at z{TILE_ZOOM}")

### Authentication

Run `planet auth init` once in a CryoCloud terminal. That writes `~/.planet.json` with
restrictive permissions, so your key never enters a notebook cell or a git commit.

In [ ]:
try:
    auth = Auth.from_file()
    print("Authenticated from ~/.planet.json")
except Exception:
    if not os.environ.get("PL_API_KEY"):
        raise SystemExit("Run `planet auth init` in a terminal, or export PL_API_KEY.")
    auth = Auth.from_key(os.environ["PL_API_KEY"])
    print("Authenticated from PL_API_KEY")

API_KEY = auth.value        # for raw tile / thumbnail HTTP requests

In [ ]:
aoi = json.load(open(AOI_PATH))
if   aoi["type"] == "FeatureCollection": aoi = aoi["features"][0]["geometry"]
elif aoi["type"] == "Feature":           aoi = aoi["geometry"]
assert aoi["type"] == "Polygon"

ring = aoi["coordinates"][0]
AOI_BBOX = (min(c[0] for c in ring), min(c[1] for c in ring),
            max(c[0] for c in ring), max(c[1] for c in ring))
HYD_LON, HYD_LAT = -125.278277, 48.814200
print(f"AOI bbox: {AOI_BBOX}")

## 2. Search — deliberately loose

Six separate June–August windows joined with an OR filter. A single
`gte=2020-06-01, lte=2025-08-31` range would sweep in every winter in between.

`cloud_cover` is a **fraction**, not a percent — passing `10` instead of `0.10` matches
everything ever acquired.

In [ ]:
search_filter = data_filter.and_filter([
    data_filter.geometry_filter(aoi),
    data_filter.range_filter("cloud_cover", lte=SEARCH_CLOUD_MAX),
    data_filter.or_filter([
        data_filter.date_range_filter("acquired",
                                      gte=datetime(y, SUMMER[0], 1),
                                      lt =datetime(y, SUMMER[1], 1))
        for y in YEARS]),
    data_filter.permission_filter(),
    data_filter.string_in_filter("quality_category", ["standard"]),
])

async def run_search():
    async with Session(auth=auth) as sess:
        res = sess.client("data").search([ITEM_TYPE],
                                         search_filter=search_filter, limit=0)
        return [i async for i in res]

items = await run_search()
json.dump(items, open(WORK / "search_results.json", "w"))
print(f"{len(items)} candidates cached (search costs no quota)")

In [ ]:
df = pd.DataFrame([{
    "id":            it["id"],
    "acquired":      pd.to_datetime(it["properties"]["acquired"]),
    "cloud_cover":   it["properties"]["cloud_cover"],
    "clear_percent": it["properties"].get("clear_percent"),
    "instrument":    it["properties"].get("instrument"),
    "sun_elevation": it["properties"].get("sun_elevation"),
    "sun_azimuth":   it["properties"].get("sun_azimuth"),
    "view_angle":    it["properties"].get("view_angle"),
    "sat_azimuth":   it["properties"].get("satellite_azimuth"),
    "thumbnail":     it["_links"].get("thumbnail"),
    "tiles_link":    it["_links"].get("tiles"),
} for it in items])

df["date"] = df["acquired"].dt.date
df["year"] = df["acquired"].dt.year
df = df.sort_values("acquired").reset_index(drop=True)

print(df.groupby("year").agg(scenes=("id","size"), days=("date","nunique")))
print(f"\ntiles link present on {df['tiles_link'].notna().sum()}/{len(df)} items")
if df["tiles_link"].isna().all():
    print("  -> set TILE_URL_TEMPLATE in stage 7; inspect items[0]['_links'] first")

print(json.dumps(items[0]["_links"], indent=2))

In [ ]:
# Route probe. Items carry no "_links.tiles" on this plan, so the tile route has to be
# supplied by hand. Probing with an arbitrary df.iloc[0] is NOT a valid test: that scene
# may not cover the target, and the tile service answers 200 with a transparent PNG for
# any tile outside a scene footprint. Probe with a scene that passes coverage, and read
# the ALPHA CHANNEL, not just the status code.
import math, httpx, numpy as np
from io import BytesIO
from PIL import Image
from shapely.geometry import shape as _shape

def deg2tile(lon, lat, z):
    n = 2 ** z
    return (int((lon + 180.0) / 360.0 * n),
            int((1.0 - math.asinh(math.tan(math.radians(lat))) / math.pi) / 2.0 * n))

# A scene that actually contains the hydrophone, so a blank result means a broken route.
_aoi_shp = _shape(aoi)
_covering = [it for it in items
             if _shape(it["geometry"]).intersection(_aoi_shp).area / _aoi_shp.area >= MIN_COVERAGE]
_probe = _covering[0]
_pid   = _probe["id"]
z      = TILE_ZOOM
x, y   = deg2tile(HYD_LON, HYD_LAT, z)

# Far outside the footprint: the negative control that shows what "no data" looks like.
_b = _shape(_probe["geometry"]).bounds
xo, yo = deg2tile(_b[2] + 0.20, _b[3] + 0.20, z)

candidates = [
    ("tiles.planet  /data/v1/PSScene/{id}/{z}/{x}/{y}.png",
     f"https://tiles.planet.com/data/v1/PSScene/{_pid}/{z}/{{X}}/{{Y}}.png"),
    ("tiles.planet  .../item-types/PSScene/items/{id}/tile/...",
     f"https://tiles.planet.com/data/v1/item-types/PSScene/items/{_pid}/tile/{z}/{{X}}/{{Y}}.png"),
    ("tiles.planet  .../item-types/PSScene/items/{id}/tiles/...",
     f"https://tiles.planet.com/data/v1/item-types/PSScene/items/{_pid}/tiles/{z}/{{X}}/{{Y}}.png"),
    ("api.planet    /data/v1/PSScene/{id}/{z}/{x}/{y}.png",
     f"https://api.planet.com/data/v1/PSScene/{_pid}/{z}/{{X}}/{{Y}}.png"),
]

def _describe(r):
    """Status code alone cannot tell imagery from nodata - decode and look."""
    if r.status_code != 200 or "image" not in r.headers.get("content-type", ""):
        return f"{r.status_code} {len(r.content):>7}B  {r.text[:40]!r}"
    a = np.array(Image.open(BytesIO(r.content)).convert("RGBA"))
    cov = (a[..., 3] > 0).mean()
    p99 = int(np.percentile(a[..., :3][a[..., 3] > 0], 99)) if cov else -1
    kind = "NODATA" if cov < 0.01 else ("flat" if p99 < 25 else "IMAGERY")
    return f"200 {len(r.content):>7}B  alpha={cov:6.1%}  p99={p99:>3}  {kind}"

print(f"probe scene {_pid}  (coverage-gated)\n")
with httpx.Client(auth=(API_KEY, ""), timeout=30, follow_redirects=True) as c:
    for label, tpl in candidates:
        for tag, (tx, ty) in (("on-target ", (x, y)), ("off-scene ", (xo, yo))):
            u = tpl.replace("{X}", str(tx)).replace("{Y}", str(ty))
            try:
                print(f"  {label[:44]:44s} {tag} {_describe(c.get(u))}")
            except Exception as e:
                print(f"  {label[:44]:44s} {tag} ERR {type(e).__name__}")
    print()

# Result on this account: only the first route serves imagery. The off-scene control
# returns 200 with an 820-byte fully transparent PNG - which is why a naive
# "did I get a 200?" check passes while the mosaic comes back black.

## 3. Gate 1 — coverage

`geometry_filter` matches on **intersects**, so a scene clipping one corner of the AOI comes
back looking identical to one covering it whole. Planet offers no "contains" filter, so this
is enforced here.

A scene covering 30% of the AOI is charged the same 100 km² as a full one — the worst value
available — and leaves nodata over exactly the water you care about.

The footprint polygon is already in the cached results, so this costs nothing.

In [ ]:
aoi_shp = shape(aoi)
df["aoi_coverage"] = df["id"].map(
    {it["id"]: shape(it["geometry"]).intersection(aoi_shp).area / aoi_shp.area
     for it in items})

full   = df["aoi_coverage"] >= MIN_COVERAGE
sliver = df["aoi_coverage"] < 0.5
print(f"full coverage : {full.sum():4d}")
print(f"partial       : {((~full) & ~sliver).sum():4d}")
print(f"slivers (<50%): {sliver.sum():4d}")
print(f"\nquota slivers would waste: {sliver.sum()*100:,} km^2 charged for "
      f"{df.loc[sliver,'aoi_coverage'].sum()*100:.0f} km^2 of usable pixels")

before = len(df)
df = df[full].copy()
print(f"\nGate 1: {before} -> {len(df)} scenes fully covering the AOI")

## 4. Gate 2 — cloud, measured on the AOI

Planet's documentation states that downloading UDM2 via the **Data API** does not count
against download quota. That is what makes this affordable.

`cloud_cover` and `clear_percent` are computed over the whole ~637 km² footprint, and your
AOI is ~15% of it, so the scene-level number tells you little. A 55%-clear scene may be
spotless over Folger Passage; a 96%-clear scene can have its one cloud sitting on the
hydrophone.

Single-band windowed reads keep peak memory near 10 MB regardless of scene size.

In [ ]:
UDM2_CLEAR_BAND = 1   # 1 clear, 2 snow, 3 shadow, 4 light haze,
                      # 5 heavy haze, 6 cloud, 7 confidence, 8 unusable-data mask

def aoi_clear_fraction(udm_path, aoi_geojson):
    '''Fraction of AOI pixels flagged clear.
    boundless=True pads outside the raster with 0 (= not clear), so a scene with
    partial coverage scores low honestly rather than returning a truncated array
    that misaligns against the geometry mask.'''
    with rasterio.open(udm_path) as src:
        geom = transform_geom("EPSG:4326", src.crs, aoi_geojson)
        win  = from_bounds(*geom_bounds(geom), src.transform)
        win  = win.round_offsets().round_lengths()
        clear = src.read(UDM2_CLEAR_BAND, window=win, boundless=True, fill_value=0)
        if clear.size == 0:
            return np.nan
        inside = geometry_mask([geom], out_shape=clear.shape,
                               transform=src.window_transform(win), invert=True)
        return float((clear[inside] == 1).mean()) if inside.any() else np.nan


async def screen_udm2(item_ids, directory=None, keep_files=False):
    '''Download + score UDM2. Costs no imagery quota. Files deleted after scoring
    unless keep_files, to protect CryoCloud disk.'''
    directory = directory or UDM_DIR
    out = {}
    async with Session(auth=auth) as sess:
        cl = sess.client("data")
        for n, iid in enumerate(item_ids, 1):
            try:
                a = await cl.get_asset(ITEM_TYPE, iid, "ortho_udm2")
                await cl.activate_asset(a)
                a = await cl.wait_asset(a, max_attempts=200)
                p = await cl.download_asset(a, directory=directory,
                                            overwrite=False, progress_bar=False)
                out[iid] = aoi_clear_fraction(p, aoi)
                if not keep_files:
                    Path(p).unlink(missing_ok=True)
            except Exception as e:
                print(f"  {iid}: {type(e).__name__} {e}")
                out[iid] = np.nan
            if n % 25 == 0:
                print(f"  screened {n}/{len(item_ids)}")
    return out

df["aoi_clear"] = df["id"].map(await screen_udm2(df["id"].tolist()))

In [ ]:
passes = df["aoi_clear"] >= CLEAR_MIN
print(f"AOI-clear vs scene clear_percent correlation: "
      f"{df[['aoi_clear','clear_percent']].corr().iloc[0,1]:.2f}")
print(f"recovered (clear on AOI, but cloud_cover > 0.10): "
      f"{(passes & (df['cloud_cover'] > 0.10)).sum()}")
print(f"rejected  (cloud_cover <= 0.10, but not clear on AOI): "
      f"{((df['cloud_cover'] <= 0.10) & ~passes).sum()}")

before = len(df)
df = df[passes].copy()
print(f"\nGate 2: {before} -> {len(df)} scenes >= {CLEAR_MIN:.0%} clear over the AOI")

## 5. Glint — review-queue ordering only

Glint is deterministic geometry, computable from metadata for free. It is **not** a gate here:
your tile budget comfortably exceeds the candidate pool, so there is no reason to discard
scenes you could simply look at. It is used to order the review queue so the scenes most
likely to show something come first.

$$\cos\Theta_g = \cos\theta_v\cos\theta_s - \sin\theta_v\sin\theta_s\cos(\phi_v-\phi_s)$$

Small $\Theta_g$ means the sensor is staring into the sun's glitter pattern, which flattens
wake contrast. Set `GLINT_FILTER` to a number only if you find glinted scenes are wasting
your review time.

In [ ]:
ts   = np.radians(90.0 - df["sun_elevation"])
tv   = np.radians(df["view_angle"].abs())
dphi = np.radians(df["sat_azimuth"] - df["sun_azimuth"])
df["glint_angle"] = np.degrees(np.arccos(np.clip(
    np.cos(tv)*np.cos(ts) - np.sin(tv)*np.sin(ts)*np.cos(dphi), -1, 1)))

print(df["glint_angle"].describe().round(1))

if GLINT_FILTER is not None:
    before = len(df)
    df = df[df["glint_angle"] > GLINT_FILTER].copy()
    print(f"\nglint filter: {before} -> {len(df)}")

# Best-looking first: clearest, then least glinted.
df = df.sort_values(["aoi_clear", "glint_angle"], ascending=False).reset_index(drop=True)
print(f"\n{len(df)} scenes queued for review")

## 6. Quicklook skim — free

Thumbnails run ~100 m/pixel, so no boat is visible. What they do show is marine fog banks and
broad glint sheets, both of which UDM2 misreads as clear over water. A quick pass here saves
tile quota and review time.

In [ ]:
def contact_sheet(rows, url_col="thumbnail", cols=6, thumb=200):
    imgs = []
    with httpx.Client(auth=(API_KEY, ""), timeout=30, follow_redirects=True) as c:
        for _, r in rows.iterrows():
            if not r.get(url_col):
                continue
            try:
                imgs.append(Image.open(BytesIO(c.get(r[url_col]).content))
                            .convert("RGB").resize((thumb, thumb)))
            except Exception:
                pass
    if not imgs:
        return None
    nrow = -(-len(imgs) // cols)
    sheet = Image.new("RGB", (cols*thumb, nrow*thumb), "black")
    for i, im in enumerate(imgs):
        sheet.paste(im, ((i % cols)*thumb, (i // cols)*thumb))
    return sheet

contact_sheet(df.head(36))

In [ ]:
FOGGED = []          # scene ids that are obviously fogged or glinted
df = df[~df["id"].isin(FOGGED)].copy()
print(f"{len(df)} scenes after visual skim")

## 7. Gate 3 — tile preview and vessel review

Tiles are metered separately from the imagery quota, so this is free against the 30-scene
budget. At z15 the ground resolution is 3.15 m/px, matching PlanetScope native.

| Object | Pixels at z15 |
|---|---|
| 7 m skiff | 2.2 |
| 12 m charter | 3.8 |
| 30 m coastal freighter | 9.5 |
| Planing wake, width | 19 |
| Planing wake, length | 127 |

This resolves **wakes and mid-size hulls**, not small boats sitting still. The inner 5 × 5 km
box is 56 tiles per scene (~1,750 previews affordable); set `PREVIEW_HALF_M = 5000` for the
full AOI at 196 tiles (~500 previews).

In [ ]:
# Verified against this account in stage 2's probe: this is the Data API XYZ tile route.
# It needs NO asset activation - every asset on these items reports status="inactive"
# and tiles still render. Confirmed 200-with-pixels on all 50 coverage-gated scenes,
# 2020-2025, both PS2.SD and PSB.SD. tiles0-3.planet.com are equivalent shards.
TILE_URL_TEMPLATE = "https://tiles.planet.com/data/v1/PSScene/{item_id}/{z}/{x}/{y}.png"

# Three distinct tile states, separable only after decoding. Byte size alone is a good
# first-pass tell because a constant-valued PNG compresses to almost nothing:
#   ~820 B, alpha == 0        -> NODATA, tile outside the scene footprint
#   1-5 kB, alpha == 1, p99<25 -> opaque but flat; no usable contrast, unreviewable
#   30-140 kB                  -> real imagery
NODATA_ALPHA_FRAC = 0.01   # below this, treat the tile as outside the footprint
FLAT_P99          = 25     # 99th pct DN over valid pixels; below this nothing is visible

def box_around(lon, lat, half_m):
    dlat = half_m / 111_206.0
    dlon = half_m / (111_320.0 * math.cos(math.radians(lat)))
    return (lon - dlon, lat - dlat, lon + dlon, lat + dlat)

def tile_grid(bbox, z):
    x0, y0 = deg2tile(bbox[0], bbox[3], z)
    x1, y1 = deg2tile(bbox[2], bbox[1], z)
    return list(range(x0, x1 + 1)), list(range(y0, y1 + 1))

PREVIEW_BOX = box_around(HYD_LON, HYD_LAT, PREVIEW_HALF_M)
xs, ys = tile_grid(PREVIEW_BOX, TILE_ZOOM)
per_scene = len(xs) * len(ys)
print(f"{2*PREVIEW_HALF_M/1000:.0f} km box -> {len(xs)}x{len(ys)} = {per_scene} tiles/scene")
print(f"affordable previews: {TILE_BUDGET // per_scene:,}   candidates: {len(df)}")

In [ ]:
def _ledger():
    if TILE_LEDGER.exists():
        return json.load(open(TILE_LEDGER))
    return {"month": datetime.now().strftime("%Y-%m"), "used": 0}

def _spend(n):
    led, now = _ledger(), datetime.now().strftime("%Y-%m")
    if led["month"] != now:            # tile quota resets monthly, no rollover
        led = {"month": now, "used": 0}
    led["used"] += n
    json.dump(led, open(TILE_LEDGER, "w"))
    return led

def tile_url(row, z, x, y):
    tpl = row.get("tiles_link") or TILE_URL_TEMPLATE
    if not tpl:
        raise RuntimeError("No tile URL: set TILE_URL_TEMPLATE in this stage.")
    return (tpl.replace("{item_id}", row["id"]).replace("{z}", str(z))
               .replace("{x}", str(x)).replace("{y}", str(y)).replace("{0}", "0"))


def classify_tile(arr):
    """NODATA / flat / imagery. A 200 proves only that the request was well-formed:
    the tile service returns a transparent PNG for anything off-footprint."""
    valid = arr[..., 3] > 0
    cov = float(valid.mean())
    if cov < NODATA_ALPHA_FRAC:
        return "nodata", cov, -1
    p99 = int(np.percentile(arr[..., :3][valid], 99))
    return ("flat" if p99 < FLAT_P99 else "imagery"), cov, p99


async def fetch_mosaic(row, bbox, z, concurrency=6, tile_px=256):
    """Fetch tiles for bbox and composite. Returns (RGBA sheet, qc dict).
    Config errors raise immediately rather than being retried into a black canvas."""
    xs, ys = tile_grid(bbox, z)
    sheet  = Image.new("RGBA", (len(xs)*tile_px, len(ys)*tile_px))
    sem    = asyncio.Semaphore(concurrency)
    qc     = {"requested": len(xs)*len(ys), "served": 0,
              "imagery": 0, "flat": 0, "nodata": 0, "failed": 0}

    tile_url(row, z, xs[0], ys[0])     # fail fast on a bad template, before any spend

    async def one(client, i, x, j, y):
        async with sem:
            url = tile_url(row, z, x, y)
            for attempt in range(4):
                try:
                    r = await client.get(url)
                except Exception:
                    await asyncio.sleep(2 ** attempt); continue
                if r.status_code == 429:                       # rate limited
                    await asyncio.sleep(2 ** attempt); continue
                if r.status_code != 200:
                    qc["failed"] += 1
                    if attempt == 0:
                        print(f"    {r.status_code} {url.rsplit('/data/v1/',1)[-1]}")
                    return
                qc["served"] += 1
                im  = Image.open(BytesIO(r.content)).convert("RGBA")
                arr = np.array(im)
                kind, _, _ = classify_tile(arr)
                qc[kind] += 1
                if kind != "nodata":
                    sheet.paste(im, (i*tile_px, j*tile_px))
                return
            qc["failed"] += 1

    async with httpx.AsyncClient(auth=(API_KEY, ""), timeout=60,
                                 follow_redirects=True) as client:
        await asyncio.gather(*[one(client, i, x, j, y)
                               for i, x in enumerate(xs) for j, y in enumerate(ys)])
    _spend(qc["served"])               # count tiles the service actually served
    return sheet, qc


def stretch(sheet, lo_pct=1.0, hi_pct=99.5):
    """Percentile stretch over valid pixels only.

    The tile service renders an 8-bit product tuned for land. Over Folger Passage the
    water occupies roughly DN 0-38 of 255 - a raw preview is visually black and a
    reviewer will score every scene 'no vessel'. Nodata stays black so it cannot be
    mistaken for dark water."""
    a     = np.array(sheet)
    valid = a[..., 3] > 0
    if not valid.any():
        return Image.fromarray(a[..., :3]), (0, 0)
    v      = a[..., :3][valid]
    lo, hi = np.percentile(v, lo_pct), np.percentile(v, hi_pct)
    out    = np.clip((a[..., :3].astype(np.float32) - lo) / max(hi - lo, 1) * 255,
                     0, 255).astype(np.uint8)
    out[~valid] = 0
    return Image.fromarray(out), (float(lo), float(hi))


async def preview_batch(rows, bbox, z, max_px=1400, quality=90):
    """One stretched JPEG per scene, tiles discarded. Scenes whose tiles carry no
    usable contrast are reported, not silently saved as a black square."""
    out, skipped = [], []
    for n, (_, r) in enumerate(rows.iterrows(), 1):
        try:
            sheet, qc = await fetch_mosaic(r, bbox, z)
            if qc["imagery"] == 0:
                skipped.append((r["id"], qc))
                print(f"  {r['id']}: no usable tiles {qc}")
                continue
            im, (lo, hi) = stretch(sheet)
            im.thumbnail((max_px, max_px))
            p = PREVIEW / f"{r['id']}.jpg"
            im.save(p, "JPEG", quality=quality)
            out.append(p)
            if qc["imagery"] < qc["requested"]:
                print(f"  {r['id']}: {qc['imagery']}/{qc['requested']} imagery "
                      f"(nodata {qc['nodata']}, flat {qc['flat']}, failed {qc['failed']})")
        except Exception as e:
            print(f"  {r['id']}: {type(e).__name__} {e}")
        if n % 20 == 0:
            print(f"  {n}/{len(rows)}   tiles used: {_ledger()['used']:,}")
    if skipped:
        print(f"\n{len(skipped)} scene(s) produced no reviewable preview")
    return out

**Test on one scene before running the batch.** Tiles are 8-bit RGB rendered with a stretch
tuned for land, and over dark water that can crush wake contrast toward black. Pick a date
with a known close vessel passage and confirm you can see it; otherwise a page of
empty-looking previews will read as "no boats" when it means "wrong stretch".

In [ ]:
test = df.iloc[[0]]
paths = await preview_batch(test, PREVIEW_BOX, TILE_ZOOM)
Image.open(paths[0]) if paths else print("no preview produced - check TILE_URL_TEMPLATE")

In [ ]:
paths = await preview_batch(df, PREVIEW_BOX, TILE_ZOOM)
led = _ledger()
print(f"\n{len(paths)} previews in {PREVIEW}")
print(f"Tiles used: {led['used']:,} / {TILE_BUDGET:,} "
      f"({TILE_BUDGET - led['used']:,} left this month)")

### Record what you see

Open the JPEGs in `planet_folger/tile_previews/` and list the ids showing a vessel or wake.

Two suggestions. Use `UNSURE` rather than forcing a binary call on an ambiguous three-pixel
blob — you can decide later whether to spend slots on those. And if you plan to run the
acoustic join, label these **before** looking at the hydrophone record, or you will
unconsciously find boats in scenes you already know were noisy.

In [ ]:
VESSEL_VISIBLE = [
    # "20230714_190112_23_2439",
]
UNSURE = [
]

df["vessel_visible"] = df["id"].isin(VESSEL_VISIBLE)
df["unsure"]         = df["id"].isin(UNSURE)

print(f"vessel visible : {df['vessel_visible'].sum()}")
print(f"unsure         : {df['unsure'].sum()}")
print(f"no vessel      : {(~df['vessel_visible'] & ~df['unsure']).sum()}")
df.to_csv(WORK / "screened_candidates.csv", index=False)

## 8. Acoustic join — optional

Off by default (`ACOUSTIC_JOIN = False`). It adds columns and prints a contingency table but
**gates nothing** — selection depends only on the three gates above.

If you do enable it, the visible-but-not-audible cell is the scientifically interesting one:
that is where acoustic detection range falls off, and it is why the AOI is wider than the
assumed detection radius.

In [ ]:
if ACOUSTIC_JOIN:
    det = pd.read_csv("hydrophone_detections.csv", parse_dates=["start_utc", "end_utc"])
    WINDOW = timedelta(minutes=15)
    df["n_detections"] = df["acquired"].apply(
        lambda t: int(((det["start_utc"] <= t + WINDOW) &
                       (det["end_utc"]   >= t - WINDOW)).sum()))
    df["audible"] = df["n_detections"] > 0
    print(pd.crosstab(df["vessel_visible"], df["audible"],
                      rownames=["visible"], colnames=["audible"]))
else:
    df["n_detections"], df["audible"] = 0, False
    print("Acoustic join disabled (ACOUSTIC_JOIN = False)")

## 9. Select and order

No scoring function. The three gates have already done the selection; all that remains is to
handle same-day duplicates and, if more scenes passed than the monthly budget allows, take
the cleanest first.

In [ ]:
sel = df[df["vessel_visible"]].copy()
print(f"{len(sel)} scenes passed all three gates")

if DEDUPE_BY_DAY:
    before = len(sel)
    sel = (sel.sort_values("aoi_clear", ascending=False)
              .groupby("date", as_index=False).head(1))
    print(f"one-per-day dedup: {before} -> {len(sel)}")

sel = sel.sort_values("aoi_clear", ascending=False)     # cleanest first
batch = sel.head(SCENES_PER_MONTH)

print(f"\nThis month: {len(batch)} scenes = {len(batch)*CHARGE_PER_SCENE:.0f} km^2 "
      f"of {MONTHLY_QUOTA:.0f}")
if len(sel) > SCENES_PER_MONTH:
    print(f"{len(sel) - len(batch)} scenes queued for following months")
print(f"AOI clear range in batch: {batch['aoi_clear'].min():.3f} - "
      f"{batch['aoi_clear'].max():.3f}")

# --------------------------------------------------------------------- SPEND GATE
CONFIRM_ORDER = False

In [ ]:
async def place_and_download(ids, name):
    out = WORK / "downloads" / name
    out.mkdir(parents=True, exist_ok=True)
    async with Session(auth=auth) as sess:
        cl = sess.client("orders")
        req = order_request.build_request(
            name=name,
            products=[order_request.product(item_ids=ids, product_bundle=BUNDLE,
                                            item_type=ITEM_TYPE)],
            tools=[order_request.clip_tool(aoi=aoi)],   # never skip: 6x saving
        )
        with reporting.StateBar(state="creating") as bar:
            order = await cl.create_order(req)
            bar.update(state="created", order_id=order["id"])
            await cl.wait(order["id"], callback=bar.update_state, max_attempts=0)
        await cl.download_order(order["id"], directory=out,
                                overwrite=False, progress_bar=True)
    return order["id"], out


if not CONFIRM_ORDER:
    print("CONFIRM_ORDER is False - nothing ordered, no quota spent.")
else:
    ids  = batch["id"].tolist()
    name = f"folger_{datetime.now():%Y%m}"
    oid, out = await place_and_download(ids, name)
    batch.to_csv(WORK / f"manifest_{name}.csv", index=False)
    print(f"Order {oid} -> {out}")

    # Single-raster bundles ship no UDM2, so re-fetch and KEEP masks for the ordered
    # scenes. Free via the Data API. They arrive as full scenes - window at read time.
    keep = out / "udm2"; keep.mkdir(exist_ok=True)
    await screen_udm2(ids, directory=keep, keep_files=True)
    print(f"UDM2 masks retained in {keep}")

## 10. Check both budgets

In [ ]:
r = httpx.get("https://api.planet.com/auth/v1/experimental/public/my/subscriptions",
              auth=(API_KEY, ""), timeout=30)
for s in r.json():
    if s.get("state") == "active":
        print(f"{s.get('plan',{}).get('name')}: "
              f"{s.get('quota_used')} / {s.get('quota_sqkm')} km^2")

led = _ledger()
print(f"\nTiles ({led['month']}): {led['used']:,} / {TILE_BUDGET:,}")

## Next steps

`manifest_*.csv` carries the UTC `acquired` timestamp for each ordered scene — the join key
back to the hydrophone record.

Both quotas reset on the calendar month and neither rolls over, so run a monthly cadence
rather than saving up. If more than 30 scenes pass the gates, the surplus stays in `sel` and
you simply re-run stage 9 next month.

**Resolved — tile access.** Items on this plan carry no `_links.tiles`, and the item
`_permissions` list contains only `assets.*:download`, so the route has to be supplied by
hand. The working one is

```
https://tiles.planet.com/data/v1/PSScene/{item_id}/{z}/{x}/{y}.png
```

authenticated with the API key as HTTP basic user (or `?api_key=`). It needs **no asset
activation** — every asset on these items reports `status="inactive"` and tiles still
render. Verified on all 50 coverage-gated scenes, 2020–2025, PS2.SD and PSB.SD: 50/50
returned imagery. `tiles0-3.planet.com` are equivalent shards.

The earlier `200 / 820 bytes` result was a **fully transparent nodata tile**, not a working
route: the probe used `df.iloc[0]`, which covers only 14% of the AOI and does not contain
the hydrophone, so the requested tile fell outside its footprint. The tile service answers
200 for any well-formed request and encodes "no data here" in the alpha channel, so status
code alone cannot distinguish a broken route from an uncovered target. Stage 7 now
classifies every tile as `nodata` / `flat` / `imagery` and the ledger counts only tiles the
service actually served.

**Still worth resolving:**

1. Confirm bare `analytic_sr` / `analytic_8b_sr` bundle names with `planet orders bundles`
   before switching off `visual`.
2. Tiles are screening only — reprojected, lossy, no radiometry. Anything entering a figure
   or a measurement comes from the ordered scene.
3. The tile renderer is tuned for land: over Folger Passage the water sits in roughly
   DN 0–38 of 255, so previews are stretched before saving. A handful of scenes
   (e.g. `20240801_192823_14_24ee`, `20250810_194613_51_253d`) render almost flat — a
   ~7 DN spread that stretching only posterizes. Stage 7 reports these rather than saving a
   black square; treat them as *not reviewed*, not as *no vessel*.
4. Validate the stretch against a date with a known vessel passage before trusting a
   negative, as stage 7's note says. The stretch makes wakes and surface texture visible,
   but it has not yet been calibrated against ground truth.